In [1]:
# ============================================================
# BiasFlow v4.1: Multi-stage / Multi-operator / Multi-fault campaign
# COMPAS Recidivism controlled fault-injection experiment
# ============================================================
# Purpose
#   - Extends the previous single feature-scaling experiment.
#   - Separates generation-stage localization from output exposure.
#   - Supports faults at data acquisition, cleaning, labeling,
#     feature engineering, training, and decision stages.
#   - Supports single faults, strength sweeps, and composite faults.
#   - Distinguishes fault-delta RCA from ordinary protected-proxy MI.
#   - Saves checkpoint CSV files after every completed run so long
#     local experiments can resume safely.
#
# Recommended use with the existing v3 script
#   - v3: repair/operator/top-k/model analysis
#   - v4: broad fault-model localization and RCA campaign
# ============================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Keep the script lightweight for long local runs.
# IPython display is optional; plain print avoids slow startup in some environments.
display = print


# ============================================================
# 1. Configuration
# ============================================================

SCHEMA_VERSION = "4.1"


CONFIG: Dict[str, Any] = {
    # ---- Paths: change these for your local machine ----
    "CSV_PATH": r"/content/drive/MyDrive/Paper_Bias/LTDD/LTDD-main/datasets/compas-scores-two-years.csv",
    "OUTPUT_PREFIX": r"/content/drive/MyDrive/Paper_Bias/results_compas_full/biasflow_v4_1_compas_fault_campaign",

    # ---- Dataset ----
    "PROTECTED_POSITIVE_VALUE": "African-American",
    "INCLUDE_PROTECTED_FEATURE": False,
    "TEST_SIZE": 0.30,

    # ---- Campaign ----
    # "screening": medium-strength representative single + multi faults
    # "strength": representative low/medium/high single-fault sweep
    # "full": screening + strength sweep
    "CAMPAIGN_MODE": "full",
    "MODELS": ["lr", "rf", "hgb", "mlp"],
    "REPEATS": 5,
    "SEED_START": 0,
    # Example: ["S3_SCALE_HOURS_M", "M1_LABEL_PLUS_SCALE"]. None runs all.
    "SCENARIO_IDS": None,

    # Derive localization/RCA metrics for these k values without retraining.
    "EVAL_K_VALUES": [1, 2, 3, 5],

    # Controlled generation/exposure signal thresholds.
    # Ranking metrics are always saved, even below threshold.
    "GENERATION_TAU": 0.01,
    "EXPOSURE_TAU": 0.005,

    # Root-cause stage selection. Among stages whose generation norm is at least
    # this fraction of the maximum, choose the earliest stage as the root.
    # This prevents a downstream imputation/processing echo from replacing the
    # stage that originally introduced a near-identical fault signal.
    "ROOT_NEAR_MAX_RATIO": 0.95,

    # Mean change in |SPD|, |AOD|, and |EOD| below this magnitude is treated
    # as practically neutral when classifying the observed fairness effect.
    "FAIRNESS_EFFECT_EPS": 1e-4,

    # Long-run safety.
    "RESUME": True,
    "RESET_OUTPUTS": False,
    "VERBOSE_FIRST_RUN": True,
    "COMPUTE_PROXY_MI": True,
    "RCA_SAVE_TOP_N": 50,

    # Optional debugging limits. Use None for the full dataset/scenario set.
    "MAX_ROWS": None,
    "MAX_SCENARIOS": None,
}


STAGE_ORDER = {
    "raw_train": 0,
    "data_acquisition": 1,
    "data_cleaning": 2,
    "label_processing": 3,
    "feature_engineering": 4,
    "training_effective": 5,
    "model_score": 6,
    "decision_threshold": 7,
}

GENERATION_STAGES = [
    "raw_train",
    "data_acquisition",
    "data_cleaning",
    "label_processing",
    "feature_engineering",
    "training_effective",
]

EXPOSURE_STAGES = ["model_score", "decision_threshold"]


# ============================================================
# 2. General utilities
# ============================================================


def stable_int_seed(*parts: Any, modulo: int = 2**32 - 1) -> int:
    text = "||".join(str(x) for x in parts)
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(digest[:16], 16) % modulo


def safe_div(num: float, den: float, default: float = np.nan) -> float:
    if den is None or den == 0 or np.isnan(den):
        return default
    return float(num) / float(den)


def safe_norm(x: Sequence[float]) -> float:
    arr = np.asarray(x, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return float(np.linalg.norm(arr))


def find_column(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        key = str(candidate).strip().lower()
        if key in lower_map:
            return lower_map[key]
    return None


def ensure_parent(path: str | Path) -> None:
    Path(path).expanduser().resolve().parent.mkdir(parents=True, exist_ok=True)


def append_csv(df: pd.DataFrame, path: str | Path) -> None:
    if df is None or len(df) == 0:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists() or path.stat().st_size == 0
    df.to_csv(path, mode="a", header=write_header, index=False, encoding="utf-8-sig")


def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = [
            "_".join(str(x) for x in col if str(x) not in {"", "None"}).strip("_")
            for col in out.columns
        ]
    return out


def json_dumps_safe(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)


# ============================================================
# 3. Adult data loading
# ============================================================


def load_adult_income(
    csv_path: str,
    protected_positive_value: str = "Male",
    exclude_protected_from_model: bool = True,
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, str, str]:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Adult CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    print(f"Loaded local CSV: {csv_path}")

    df.columns = [str(c).strip() for c in df.columns]
    df = df.replace("?", np.nan)

    target_col = find_column(df, ["Probability", "income", "class", "target", "label", "y"])
    if target_col is None:
        raise KeyError(f"Target column not found. Columns: {df.columns.tolist()}")

    protected_col = find_column(df, ["sex", "SEX"])
    if protected_col is None:
        raise KeyError(f"Protected column not found. Columns: {df.columns.tolist()}")

    y_text = df[target_col].astype(str).str.strip()
    y = y_text.str.contains(">50K", regex=False).astype(int).to_numpy()

    protected_text = df[protected_col].astype(str).str.strip().str.lower()
    protected_value = str(protected_positive_value).strip().lower()
    protected = (protected_text == protected_value).astype(int).to_numpy()

    drop_cols = [target_col]
    if exclude_protected_from_model:
        drop_cols.append(protected_col)

    X = df.drop(columns=list(set(drop_cols)), errors="ignore").copy()

    for col in X.columns:
        converted = pd.to_numeric(X[col], errors="coerce")
        if converted.notna().sum() >= max(1, int(0.8 * len(converted))):
            X[col] = converted

    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    if categorical_cols:
        X = pd.get_dummies(X, columns=categorical_cols, dummy_na=True, drop_first=False)

    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

    print("\n================ DATASET INFO ================")
    print("Dataset: Adult Income")
    print("Target column:", target_col)
    print("Protected column:", protected_col)
    print("Protected positive value:", protected_positive_value)
    print("Target class count [<=50K, >50K]:", np.bincount(y.astype(int)))
    print("Protected group count [0,1]:", np.bincount(protected.astype(int)))
    print("X shape:", X.shape)
    print("First 10 features:", X.columns.tolist()[:10])

    return X, y.astype(int), protected.astype(int), target_col, protected_col





def load_compas_recidivism(
    csv_path: str,
    protected_positive_value: str = "African-American",
    exclude_protected_from_model: bool = True,
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, str, str]:
    """Load COMPAS two-year recidivism dataset.

    Target:
      two_year_recid == 0 is treated as the favorable positive outcome
      (no recidivism within two years). Thus y=1 means a favorable outcome.

    Protected attribute:
      race is binarized by default to African-American vs Caucasian.
      A=1 corresponds to protected_positive_value, which defaults to
      African-American. Rows outside the two selected race groups are removed
      for the controlled campaign.

    Feature policy:
      The model uses pre-assessment or charge-related covariates only. Direct
      identifiers, names, dates, post-recidivism fields, and COMPAS score
      outputs such as decile_score and score_text are excluded to avoid leakage.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"COMPAS CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    print(f"Loaded local CSV: {csv_path}")

    df.columns = [str(c).strip() for c in df.columns]
    df = df.replace("?", np.nan)

    target_col = find_column(df, ["two_year_recid", "is_recid", "target", "label", "y"])
    if target_col is None:
        raise KeyError(f"Target column not found. Columns: {df.columns.tolist()}")

    protected_col = find_column(df, ["race", "Race", "RACE"])
    if protected_col is None:
        raise KeyError(f"Race column not found for protected attribute. Columns: {df.columns.tolist()}")

    # Standard COMPAS-style filtering: focus on African-American and Caucasian
    # groups and keep records close to the COMPAS screening date when available.
    race_text = df[protected_col].astype(str).str.strip()
    selected_races = ["African-American", "Caucasian"]
    df = df[race_text.isin(selected_races)].copy()

    days_col = find_column(df, ["days_b_screening_arrest"])
    if days_col is not None:
        days = pd.to_numeric(df[days_col], errors="coerce")
        df = df[days.notna() & (days.abs() <= 30)].copy()

    score_text_col = find_column(df, ["score_text"])
    if score_text_col is not None:
        df = df[df[score_text_col].astype(str).str.strip() != "N/A"].copy()

    charge_degree_col = find_column(df, ["c_charge_degree"])
    if charge_degree_col is not None:
        df = df[df[charge_degree_col].astype(str).str.strip() != "O"].copy()

    # Favorable positive outcome: no two-year recidivism.
    y_raw = pd.to_numeric(df[target_col], errors="coerce")
    if y_raw.isna().any():
        raise ValueError(f"Target column {target_col} contains non-numeric values after filtering.")
    y = (y_raw.astype(int) == 0).astype(int).to_numpy()

    ppv = str(protected_positive_value).strip().lower()
    race_text = df[protected_col].astype(str).str.strip()
    if ppv in {"african-american", "african_american", "black", "aa", "1"}:
        protected = (race_text == "African-American").astype(int).to_numpy()
        protected_label = "African-American"
    elif ppv in {"caucasian", "white", "0"}:
        protected = (race_text == "Caucasian").astype(int).to_numpy()
        protected_label = "Caucasian"
    else:
        raise ValueError(
            "protected_positive_value should be African-American or Caucasian for this COMPAS loader"
        )

    # Use only pre-assessment / charge-related features. Exclude identifiers,
    # names, raw dates, post-outcome recidivism fields, and COMPAS score outputs.
    feature_candidates = [
        "sex",
        "age",
        "age_cat",
        "juv_fel_count",
        "juv_misd_count",
        "juv_other_count",
        "priors_count",
        "days_b_screening_arrest",
        "c_days_from_compas",
        "c_charge_degree",
        "c_charge_desc",
    ]
    feature_cols = [c for c in feature_candidates if c in df.columns]
    if not feature_cols:
        raise ValueError("No usable COMPAS feature columns were found.")

    if (not exclude_protected_from_model) and protected_col not in feature_cols:
        feature_cols.append(protected_col)

    X = df[feature_cols].copy()

    # Convert numeric-looking columns and one-hot encode categorical columns.
    for col in X.columns:
        converted = pd.to_numeric(X[col], errors="coerce")
        if converted.notna().sum() >= max(1, int(0.8 * len(converted))):
            X[col] = converted

    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    if categorical_cols:
        X = pd.get_dummies(X, columns=categorical_cols, dummy_na=True, drop_first=False)

    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

    print("\n================ DATASET INFO ================")
    print("Dataset: COMPAS Recidivism")
    print("Target column:", target_col)
    print("Target positive class: two_year_recid == 0 / no recidivism")
    print("Protected column:", protected_col)
    print("Protected positive value:", protected_label)
    print("Rows after COMPAS filtering:", len(X))
    print("Target class count [recidivism/unfavorable, no-recid/favorable]:", np.bincount(y.astype(int)))
    print("Protected group count [0,1]:", np.bincount(protected.astype(int)))
    print("X shape:", X.shape)
    print("First 10 features:", X.columns.tolist()[:10])

    return X.reset_index(drop=True), y.astype(int), protected.astype(int), target_col, protected_col


# ============================================================
# 4. Fault model
# ============================================================


@dataclass(frozen=True)
class FaultSpec:
    fault_id: str
    stage: str
    operator: str
    feature: Optional[str] = None
    group: int = 1
    rate: float = 0.0
    factor: float = 1.0
    shift_std: float = 0.0
    quantile: float = 0.75
    from_label: Optional[int] = None
    to_label: Optional[int] = None
    threshold_a0: Optional[float] = None
    threshold_a1: Optional[float] = None
    score_offset: float = 0.0
    condition_feature: Optional[str] = None
    condition_op: Optional[str] = None
    condition_value: Optional[float] = None
    note: str = ""

    def as_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class FaultScenario:
    scenario_id: str
    description: str
    faults: Tuple[FaultSpec, ...] = field(default_factory=tuple)
    true_generation_stages: Tuple[str, ...] = field(default_factory=tuple)
    true_exposure_stages: Tuple[str, ...] = field(default_factory=tuple)

    # Primary carriers are direct injected causes and are used for the main RCA
    # accuracy metrics. Secondary carriers are expected downstream/side effects
    # (for example, protected prevalence after selective sampling) and are
    # evaluated separately so a correct top-1 cause is not unfairly penalized.
    true_carriers: Tuple[str, ...] = field(default_factory=tuple)
    secondary_carriers: Tuple[str, ...] = field(default_factory=tuple)

    category: str = "single"
    strength: str = "medium"
    expected_fault_role: str = "unspecified"

    @property
    def primary_carriers(self) -> Tuple[str, ...]:
        return self.true_carriers

    def as_dict(self) -> Dict[str, Any]:
        return {
            "schema_version": SCHEMA_VERSION,
            "scenario_id": self.scenario_id,
            "description": self.description,
            "faults": [f.as_dict() for f in self.faults],
            "true_generation_stages": list(self.true_generation_stages),
            "true_exposure_stages": list(self.true_exposure_stages),
            "true_primary_carriers": list(self.true_carriers),
            "true_secondary_carriers": list(self.secondary_carriers),
            # Backward-compatible alias.
            "true_carriers": list(self.true_carriers),
            "category": self.category,
            "strength": self.strength,
            "expected_fault_role": self.expected_fault_role,
        }


def fault(
    fault_id: str,
    stage: str,
    operator: str,
    **kwargs: Any,
) -> FaultSpec:
    return FaultSpec(fault_id=fault_id, stage=stage, operator=operator, **kwargs)




def make_screening_scenarios() -> List[FaultScenario]:
    """Representative medium-strength campaign for the COMPAS dataset.

    Protected group A=1 is African-American by default. The favorable label is
    no two-year recidivism. Direct feature faults use priors_count, while
    missingness uses days_b_screening_arrest.
    """
    return [
        FaultScenario(
            "CLEAN_CONTROL",
            "Negative control without injected fault.",
            faults=tuple(),
            category="control",
            strength="none",
            expected_fault_role="control",
        ),

        # ----------------------------------------------------
        # S1: data acquisition / selection
        # ----------------------------------------------------
        FaultScenario(
            "S1_UNDERSAMPLE_A1Y1_M",
            "Fairness-amplifying selection fault: remove 30% of A=1 positive training instances.",
            faults=(fault(
                "undersample_a1y1",
                "data_acquisition",
                "undersample_group_label",
                group=1,
                from_label=1,
                rate=0.30,
            ),),
            true_generation_stages=("data_acquisition",),
            true_carriers=("label_gap",),
            secondary_carriers=("protected_prevalence",),
            category="single_amplifying",
            strength="medium",
            expected_fault_role="amplifying",
        ),
        FaultScenario(
            "S1_UNDERSAMPLE_A0Y1_COMP_M",
            "Compensating selection fault: remove 30% of A=0 positive training instances.",
            faults=(fault(
                "undersample_a0y1_comp",
                "data_acquisition",
                "undersample_group_label",
                group=0,
                from_label=1,
                rate=0.30,
            ),),
            true_generation_stages=("data_acquisition",),
            true_carriers=("label_gap",),
            secondary_carriers=("protected_prevalence",),
            category="single_compensating",
            strength="medium",
            expected_fault_role="compensating",
        ),
        FaultScenario(
            "S1_CONDITIONAL_SELECTION_M",
            "Remove 40% of A=1 high-prior training instances.",
            faults=(fault(
                "conditional_selection_priors",
                "data_acquisition",
                "conditional_selection",
                feature="priors_count",
                condition_feature="priors_count",
                condition_op="gt",
                condition_value=2.0,
                group=1,
                rate=0.40,
            ),),
            true_generation_stages=("data_acquisition",),
            true_carriers=("mean::priors_count",),
            secondary_carriers=("protected_prevalence",),
            category="single_structural",
            strength="medium",
            expected_fault_role="data_distribution_shift",
        ),

        # ----------------------------------------------------
        # S2: cleaning and labels
        # ----------------------------------------------------
        FaultScenario(
            "S2_MISSING_SCREENING_DAYS_M",
            "Set days_b_screening_arrest missing for 30% of A=1 instances.",
            faults=(fault(
                "missing_screening_days",
                "data_cleaning",
                "group_missingness",
                feature="days_b_screening_arrest",
                group=1,
                rate=0.30,
            ),),
            true_generation_stages=("data_cleaning",),
            true_carriers=("missing::days_b_screening_arrest",),
            category="single_latent",
            strength="medium",
            expected_fault_role="latent_or_model_dependent",
        ),
        FaultScenario(
            "S2_LABEL_FLIP_A1Y1_M",
            "Fairness-amplifying label fault: flip 15% of A=1 positive labels from 1 to 0.",
            faults=(fault(
                "label_flip_a1y1",
                "label_processing",
                "group_label_flip",
                group=1,
                from_label=1,
                to_label=0,
                rate=0.15,
            ),),
            true_generation_stages=("label_processing",),
            true_carriers=("label_gap",),
            category="single_amplifying",
            strength="medium",
            expected_fault_role="amplifying",
        ),
        FaultScenario(
            "S2_LABEL_FLIP_A0Y1_COMP_M",
            "Compensating label corruption: flip 15% of A=0 positive labels from 1 to 0.",
            faults=(fault(
                "label_flip_a0y1_comp",
                "label_processing",
                "group_label_flip",
                group=0,
                from_label=1,
                to_label=0,
                rate=0.15,
            ),),
            true_generation_stages=("label_processing",),
            true_carriers=("label_gap",),
            category="single_compensating",
            strength="medium",
            expected_fault_role="compensating",
        ),

        # ----------------------------------------------------
        # S3: feature engineering
        # ----------------------------------------------------
        FaultScenario(
            "S3_SCALE_PRIORS_M",
            "Multiply priors_count by 1.5 for A=1.",
            faults=(fault(
                "scale_priors",
                "feature_engineering",
                "group_scale",
                feature="priors_count",
                group=1,
                factor=1.50,
            ),),
            true_generation_stages=("feature_engineering",),
            true_carriers=("mean::priors_count",),
            category="single_amplifying",
            strength="medium",
            expected_fault_role="amplifying",
        ),
        FaultScenario(
            "S3_SHIFT_PRIORS_M",
            "Add 0.5 training standard deviations to priors_count for A=1.",
            faults=(fault(
                "shift_priors",
                "feature_engineering",
                "group_shift_std",
                feature="priors_count",
                group=1,
                shift_std=0.50,
            ),),
            true_generation_stages=("feature_engineering",),
            true_carriers=("mean::priors_count",),
            category="single_amplifying",
            strength="medium",
            expected_fault_role="amplifying",
        ),
        FaultScenario(
            "S3_CLIP_PRIORS_M",
            "Clip A=1 priors_count at the global 70th percentile.",
            faults=(fault(
                "clip_priors",
                "feature_engineering",
                "group_upper_clip",
                feature="priors_count",
                group=1,
                quantile=0.70,
            ),),
            true_generation_stages=("feature_engineering",),
            true_carriers=("mean::priors_count",),
            category="single_nonlinear",
            strength="medium",
            expected_fault_role="model_dependent",
        ),

        # ----------------------------------------------------
        # S4: effective training distribution
        # ----------------------------------------------------
        FaultScenario(
            "S4_WEIGHT_A1Y1_M",
            "Fairness-amplifying training fault: down-weight A=1 positive instances to 0.5.",
            faults=(fault(
                "weight_a1y1",
                "training_effective",
                "group_label_weight",
                group=1,
                from_label=1,
                factor=0.50,
            ),),
            true_generation_stages=("training_effective",),
            true_carriers=("weight_gap",),
            secondary_carriers=("label_gap",),
            category="single_amplifying",
            strength="medium",
            expected_fault_role="amplifying",
        ),
        FaultScenario(
            "S4_WEIGHT_A0Y1_COMP_M",
            "Compensating training fault: down-weight A=0 positive instances to 0.5.",
            faults=(fault(
                "weight_a0y1_comp",
                "training_effective",
                "group_label_weight",
                group=0,
                from_label=1,
                factor=0.50,
            ),),
            true_generation_stages=("training_effective",),
            true_carriers=("weight_gap",),
            secondary_carriers=("label_gap",),
            category="single_compensating",
            strength="medium",
            expected_fault_role="compensating",
        ),

        # ----------------------------------------------------
        # S5: model-score / decision-policy faults
        # ----------------------------------------------------
        FaultScenario(
            "S5_THRESHOLD_A1_HIGH_M",
            "Fairness-amplifying decision fault: thresholds 0.50 for A=0 and 0.60 for A=1.",
            faults=(fault(
                "threshold_a1_high",
                "decision_threshold",
                "group_threshold",
                threshold_a0=0.50,
                threshold_a1=0.60,
            ),),
            true_exposure_stages=("decision_threshold",),
            true_carriers=("decision::SPD", "decision::EOD", "decision::FPR_gap"),
            category="single_amplifying",
            strength="medium",
            expected_fault_role="amplifying",
        ),
        FaultScenario(
            "S5_THRESHOLD_A0_HIGH_COMP_M",
            "Compensating decision policy: thresholds 0.60 for A=0 and 0.50 for A=1.",
            faults=(fault(
                "threshold_a0_high_comp",
                "decision_threshold",
                "group_threshold",
                threshold_a0=0.60,
                threshold_a1=0.50,
            ),),
            true_exposure_stages=("decision_threshold",),
            true_carriers=("decision::SPD", "decision::EOD", "decision::FPR_gap"),
            category="single_compensating",
            strength="medium",
            expected_fault_role="compensating",
        ),
        FaultScenario(
            "S5_SCORE_OFFSET_A1_NEG_M",
            "Fairness-amplifying score fault: subtract 0.10 from A=1 scores.",
            faults=(fault(
                "score_offset_a1_neg",
                "model_score",
                "group_score_offset",
                group=1,
                score_offset=-0.10,
            ),),
            true_exposure_stages=("model_score",),
            true_carriers=("score::mean_gap", "score::brier_gap"),
            secondary_carriers=("score::positive_label_gap", "score::negative_label_gap"),
            category="single_amplifying",
            strength="medium",
            expected_fault_role="amplifying",
        ),
        FaultScenario(
            "S5_SCORE_OFFSET_A0_NEG_COMP_M",
            "Compensating score fault: subtract 0.10 from A=0 scores.",
            faults=(fault(
                "score_offset_a0_neg_comp",
                "model_score",
                "group_score_offset",
                group=0,
                score_offset=-0.10,
            ),),
            true_exposure_stages=("model_score",),
            true_carriers=("score::mean_gap", "score::brier_gap"),
            secondary_carriers=("score::positive_label_gap", "score::negative_label_gap"),
            category="single_compensating",
            strength="medium",
            expected_fault_role="compensating",
        ),

        # ----------------------------------------------------
        # Multi-fault cases
        # ----------------------------------------------------
        FaultScenario(
            "M1_LABEL_PLUS_SCALE",
            "Distributed amplifying faults: A=1 positive label flip plus A=1 priors_count scaling.",
            faults=(
                fault(
                    "label_flip_a1y1",
                    "label_processing",
                    "group_label_flip",
                    group=1,
                    from_label=1,
                    to_label=0,
                    rate=0.15,
                ),
                fault(
                    "scale_priors",
                    "feature_engineering",
                    "group_scale",
                    feature="priors_count",
                    group=1,
                    factor=1.50,
                ),
            ),
            true_generation_stages=("label_processing", "feature_engineering"),
            true_carriers=("label_gap", "mean::priors_count"),
            category="multi_distributed",
            strength="medium",
            expected_fault_role="amplifying",
        ),
        FaultScenario(
            "M2_MISSING_PLUS_SCALE",
            "Distributed feature faults: missing days_b_screening_arrest plus priors_count scaling.",
            faults=(
                fault(
                    "missing_screening_days",
                    "data_cleaning",
                    "group_missingness",
                    feature="days_b_screening_arrest",
                    group=1,
                    rate=0.30,
                ),
                fault(
                    "scale_priors",
                    "feature_engineering",
                    "group_scale",
                    feature="priors_count",
                    group=1,
                    factor=1.50,
                ),
            ),
            true_generation_stages=("data_cleaning", "feature_engineering"),
            true_carriers=("missing::days_b_screening_arrest", "mean::priors_count"),
            category="multi_distributed",
            strength="medium",
            expected_fault_role="mixed_or_model_dependent",
        ),
        FaultScenario(
            "M3_SCALE_PLUS_MASKING_THRESHOLD",
            "A=1 priors_count scaling followed by an A=0-high threshold that can mask the output gap.",
            faults=(
                fault(
                    "scale_priors",
                    "feature_engineering",
                    "group_scale",
                    feature="priors_count",
                    group=1,
                    factor=1.50,
                ),
                fault(
                    "masking_threshold_a0_high",
                    "decision_threshold",
                    "group_threshold",
                    threshold_a0=0.60,
                    threshold_a1=0.50,
                ),
            ),
            true_generation_stages=("feature_engineering",),
            true_exposure_stages=("decision_threshold",),
            true_carriers=("mean::priors_count", "decision::SPD", "decision::EOD"),
            category="multi_masking",
            strength="medium",
            expected_fault_role="masking",
        ),
        FaultScenario(
            "M4_WEAK_INTERACTION",
            "Two individually weak faults: 10% missingness plus 1.1x priors_count scaling.",
            faults=(
                fault(
                    "weak_missing_screening_days",
                    "data_cleaning",
                    "group_missingness",
                    feature="days_b_screening_arrest",
                    group=1,
                    rate=0.10,
                ),
                fault(
                    "weak_scale_priors",
                    "feature_engineering",
                    "group_scale",
                    feature="priors_count",
                    group=1,
                    factor=1.10,
                ),
            ),
            true_generation_stages=("data_cleaning", "feature_engineering"),
            true_carriers=("missing::days_b_screening_arrest", "mean::priors_count"),
            category="multi_weak_interaction",
            strength="low+low",
            expected_fault_role="interaction",
        ),
    ]



def make_strength_scenarios() -> List[FaultScenario]:
    """Low/medium/high sweep using COMPAS fields."""
    scenarios: List[FaultScenario] = []

    levels = [
        ("L", "low", 0.10, 1.10, 0.05, 0.80, 0.05),
        ("M", "medium", 0.30, 1.30, 0.15, 0.50, 0.10),
        ("H", "high", 0.50, 1.50, 0.30, 0.20, 0.20),
    ]

    for suffix, label, miss_rate, scale_factor, flip_rate, weight_factor, threshold_gap in levels:
        scenarios.extend([
            FaultScenario(
                f"SWEEP_S2_MISSING_{suffix}",
                f"Strength sweep: A=1 days_b_screening_arrest missingness {miss_rate:.0%}.",
                faults=(fault(
                    f"missing_{suffix}",
                    "data_cleaning",
                    "group_missingness",
                    feature="days_b_screening_arrest",
                    group=1,
                    rate=miss_rate,
                ),),
                true_generation_stages=("data_cleaning",),
                true_carriers=("missing::days_b_screening_arrest",),
                category="strength_sweep",
                strength=label,
                expected_fault_role="latent_or_model_dependent",
            ),
            FaultScenario(
                f"SWEEP_S2_LABEL_{suffix}",
                f"Strength sweep: A=1 positive label flip {flip_rate:.0%}.",
                faults=(fault(
                    f"label_{suffix}",
                    "label_processing",
                    "group_label_flip",
                    group=1,
                    from_label=1,
                    to_label=0,
                    rate=flip_rate,
                ),),
                true_generation_stages=("label_processing",),
                true_carriers=("label_gap",),
                category="strength_sweep",
                strength=label,
                expected_fault_role="amplifying",
            ),
            FaultScenario(
                f"SWEEP_S3_SCALE_{suffix}",
                f"Strength sweep: A=1 priors_count scaling factor {scale_factor:.2f}.",
                faults=(fault(
                    f"scale_{suffix}",
                    "feature_engineering",
                    "group_scale",
                    feature="priors_count",
                    group=1,
                    factor=scale_factor,
                ),),
                true_generation_stages=("feature_engineering",),
                true_carriers=("mean::priors_count",),
                category="strength_sweep",
                strength=label,
                expected_fault_role="amplifying",
            ),
            FaultScenario(
                f"SWEEP_S4_WEIGHT_{suffix}",
                f"Strength sweep: A=1,y=1 sample weight factor {weight_factor:.2f}.",
                faults=(fault(
                    f"weight_{suffix}",
                    "training_effective",
                    "group_label_weight",
                    group=1,
                    from_label=1,
                    factor=weight_factor,
                ),),
                true_generation_stages=("training_effective",),
                true_carriers=("weight_gap",),
                secondary_carriers=("label_gap",),
                category="strength_sweep",
                strength=label,
                expected_fault_role="amplifying",
            ),
            FaultScenario(
                f"SWEEP_S5_THRESHOLD_{suffix}",
                f"Strength sweep: A=1 threshold raised by {threshold_gap:.2f}.",
                faults=(fault(
                    f"threshold_{suffix}",
                    "decision_threshold",
                    "group_threshold",
                    threshold_a0=0.50,
                    threshold_a1=0.50 + threshold_gap,
                ),),
                true_exposure_stages=("decision_threshold",),
                true_carriers=("decision::SPD", "decision::EOD"),
                secondary_carriers=("decision::FPR_gap",),
                category="strength_sweep",
                strength=label,
                expected_fault_role="amplifying",
            ),
        ])

    return scenarios


def build_campaign(mode: str) -> List[FaultScenario]:
    mode = str(mode).lower().strip()
    if mode == "screening":
        return make_screening_scenarios()
    if mode == "strength":
        return make_strength_scenarios()
    if mode == "full":
        # Deduplicate by scenario id.
        all_scenarios = make_screening_scenarios() + make_strength_scenarios()
        seen: set[str] = set()
        out: List[FaultScenario] = []
        for scenario in all_scenarios:
            if scenario.scenario_id not in seen:
                seen.add(scenario.scenario_id)
                out.append(scenario)
        return out
    raise ValueError(f"Unknown CAMPAIGN_MODE: {mode}")


# ============================================================
# 5. Fault application
# ============================================================


def faults_at(scenario: FaultScenario, stage: str) -> List[FaultSpec]:
    return [f for f in scenario.faults if f.stage == stage]


def condition_mask(X: pd.DataFrame, spec: FaultSpec) -> np.ndarray:
    if spec.condition_feature is None:
        return np.ones(len(X), dtype=bool)
    if spec.condition_feature not in X.columns:
        raise KeyError(f"Condition feature not found: {spec.condition_feature}")

    values = pd.to_numeric(X[spec.condition_feature], errors="coerce").to_numpy(dtype=float)
    op = spec.condition_op
    value = float(spec.condition_value)

    if op == "lt":
        return values < value
    if op == "le":
        return values <= value
    if op == "gt":
        return values > value
    if op == "ge":
        return values >= value
    if op == "eq":
        return values == value
    raise ValueError(f"Unknown condition_op: {op}")


def apply_acquisition_faults(
    X: pd.DataFrame,
    y: np.ndarray,
    protected: np.ndarray,
    specs: Sequence[FaultSpec],
    rng: np.random.Generator,
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, Dict[str, Any]]:
    Xn = X.copy().reset_index(drop=True)
    yn = np.asarray(y).astype(int).copy()
    pn = np.asarray(protected).astype(int).copy()
    log: Dict[str, Any] = {"removed": 0, "operators": []}

    for spec in specs:
        if spec.operator not in {"undersample_group_label", "conditional_selection"}:
            raise ValueError(f"Unsupported acquisition operator: {spec.operator}")

        mask = pn == int(spec.group)
        if spec.from_label is not None:
            mask &= yn == int(spec.from_label)
        if spec.operator == "conditional_selection":
            mask &= condition_mask(Xn, spec)

        candidates = np.flatnonzero(mask)
        n_remove = int(round(float(spec.rate) * len(candidates)))
        n_remove = min(max(n_remove, 0), len(candidates))

        remove_idx = (
            rng.choice(candidates, size=n_remove, replace=False)
            if n_remove > 0 else np.array([], dtype=int)
        )
        keep = np.ones(len(Xn), dtype=bool)
        keep[remove_idx] = False

        Xn = Xn.loc[keep].reset_index(drop=True)
        yn = yn[keep]
        pn = pn[keep]

        log["removed"] += int(n_remove)
        log["operators"].append({"fault_id": spec.fault_id, "removed": int(n_remove)})

    return Xn, yn, pn, log


def apply_cleaning_faults(
    X: pd.DataFrame,
    protected: np.ndarray,
    specs: Sequence[FaultSpec],
    rng: np.random.Generator,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    Xn = X.copy().reset_index(drop=True)
    p = np.asarray(protected).astype(int)
    log: Dict[str, Any] = {"set_missing": 0, "operators": []}

    for spec in specs:
        if spec.operator != "group_missingness":
            raise ValueError(f"Unsupported cleaning operator: {spec.operator}")
        if spec.feature not in Xn.columns:
            raise KeyError(f"Missingness feature not found: {spec.feature}")

        candidates = np.flatnonzero(p == int(spec.group))
        n_change = int(round(float(spec.rate) * len(candidates)))
        n_change = min(max(n_change, 0), len(candidates))
        chosen = (
            rng.choice(candidates, size=n_change, replace=False)
            if n_change > 0 else np.array([], dtype=int)
        )
        Xn.loc[chosen, spec.feature] = np.nan
        log["set_missing"] += int(n_change)
        log["operators"].append({"fault_id": spec.fault_id, "set_missing": int(n_change)})

    return Xn, log


def apply_label_faults(
    y: np.ndarray,
    protected: np.ndarray,
    specs: Sequence[FaultSpec],
    rng: np.random.Generator,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    yn = np.asarray(y).astype(int).copy()
    p = np.asarray(protected).astype(int)
    log: Dict[str, Any] = {"flipped": 0, "operators": []}

    for spec in specs:
        if spec.operator != "group_label_flip":
            raise ValueError(f"Unsupported label operator: {spec.operator}")
        if spec.from_label is None or spec.to_label is None:
            raise ValueError("group_label_flip requires from_label and to_label")

        candidates = np.flatnonzero(
            (p == int(spec.group)) & (yn == int(spec.from_label))
        )
        n_change = int(round(float(spec.rate) * len(candidates)))
        n_change = min(max(n_change, 0), len(candidates))
        chosen = (
            rng.choice(candidates, size=n_change, replace=False)
            if n_change > 0 else np.array([], dtype=int)
        )
        yn[chosen] = int(spec.to_label)
        log["flipped"] += int(n_change)
        log["operators"].append({"fault_id": spec.fault_id, "flipped": int(n_change)})

    return yn, log


class FeatureFaultTransformer:
    def __init__(self, specs: Sequence[FaultSpec]):
        self.specs = list(specs)
        self.params: Dict[str, Dict[str, float]] = {}

    def fit(self, X: pd.DataFrame) -> "FeatureFaultTransformer":
        self.params = {}
        for spec in self.specs:
            if spec.feature is None or spec.feature not in X.columns:
                raise KeyError(f"Feature-fault target not found: {spec.feature}")
            values = pd.to_numeric(X[spec.feature], errors="coerce").to_numpy(dtype=float)
            finite = values[np.isfinite(values)]
            if len(finite) == 0:
                finite = np.array([0.0])
            self.params[spec.fault_id] = {
                "std": float(np.std(finite) + 1e-8),
                "clip_value": float(np.quantile(finite, float(spec.quantile))),
            }
        return self

    def transform(self, X: pd.DataFrame, protected: np.ndarray) -> pd.DataFrame:
        Xn = X.copy().reset_index(drop=True)
        p = np.asarray(protected).astype(int)

        for spec in self.specs:
            feature = str(spec.feature)
            values = pd.to_numeric(Xn[feature], errors="coerce").fillna(0).to_numpy(dtype=float)
            mask = p == int(spec.group)

            if spec.operator == "group_scale":
                values[mask] = values[mask] * float(spec.factor)
            elif spec.operator == "group_shift_std":
                shift = float(spec.shift_std) * self.params[spec.fault_id]["std"]
                values[mask] = values[mask] + shift
            elif spec.operator == "group_upper_clip":
                clip_value = self.params[spec.fault_id]["clip_value"]
                values[mask] = np.minimum(values[mask], clip_value)
            else:
                raise ValueError(f"Unsupported feature operator: {spec.operator}")

            Xn[feature] = values

        return Xn.replace([np.inf, -np.inf], np.nan)


def build_sample_weights(
    y: np.ndarray,
    protected: np.ndarray,
    specs: Sequence[FaultSpec],
) -> Tuple[np.ndarray, Dict[str, Any]]:
    y = np.asarray(y).astype(int)
    p = np.asarray(protected).astype(int)
    weights = np.ones(len(y), dtype=float)
    log: Dict[str, Any] = {"weighted": 0, "operators": []}

    for spec in specs:
        if spec.operator not in {"group_weight", "group_label_weight"}:
            raise ValueError(f"Unsupported training operator: {spec.operator}")
        mask = p == int(spec.group)
        if spec.operator == "group_label_weight":
            if spec.from_label is None:
                raise ValueError("group_label_weight requires from_label")
            mask &= y == int(spec.from_label)
        weights[mask] *= float(spec.factor)
        log["weighted"] += int(np.sum(mask))
        log["operators"].append({
            "fault_id": spec.fault_id,
            "weighted": int(np.sum(mask)),
            "factor": float(spec.factor),
        })

    return weights, log


def apply_score_faults(
    scores: np.ndarray,
    protected: np.ndarray,
    specs: Sequence[FaultSpec],
) -> np.ndarray:
    out = np.asarray(scores, dtype=float).copy()
    p = np.asarray(protected).astype(int)
    for spec in specs:
        if spec.operator != "group_score_offset":
            raise ValueError(f"Unsupported score operator: {spec.operator}")
        out[p == int(spec.group)] += float(spec.score_offset)
    return np.clip(out, 0.0, 1.0)


def apply_threshold_faults(
    scores: np.ndarray,
    protected: np.ndarray,
    specs: Sequence[FaultSpec],
    default_threshold: float = 0.5,
) -> np.ndarray:
    p = np.asarray(protected).astype(int)
    threshold_a0 = float(default_threshold)
    threshold_a1 = float(default_threshold)

    for spec in specs:
        if spec.operator != "group_threshold":
            raise ValueError(f"Unsupported threshold operator: {spec.operator}")
        if spec.threshold_a0 is not None:
            threshold_a0 = float(spec.threshold_a0)
        if spec.threshold_a1 is not None:
            threshold_a1 = float(spec.threshold_a1)

    thresholds = np.where(p == 1, threshold_a1, threshold_a0)
    return (np.asarray(scores, dtype=float) >= thresholds).astype(int)


# ============================================================
# 6. Models and fitting
# ============================================================


def make_model(model_name: str, random_state: int):
    name = str(model_name).lower().strip()
    if name in {"lr", "logistic", "logistic_regression"}:
        return LogisticRegression(
            max_iter=1000,
            solver="liblinear",
            random_state=random_state,
        )
    if name in {"rf", "random_forest", "randomforest"}:
        return RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=random_state,
        )
    if name in {"hgb", "hist_gradient_boosting", "gbdt"}:
        return HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            max_leaf_nodes=31,
            random_state=random_state,
        )
    if name in {"mlp", "neural", "neural_network"}:
        return MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=100,
            early_stopping=True,
            n_iter_no_change=10,
            random_state=random_state,
        )
    raise ValueError(f"Unknown model: {model_name}")


def fit_model_with_weights(
    model,
    X: pd.DataFrame,
    y: np.ndarray,
    sample_weight: np.ndarray,
    random_state: int,
) -> Tuple[Any, str]:
    weights = np.asarray(sample_weight, dtype=float)
    if np.allclose(weights, 1.0):
        model.fit(X, y)
        return model, "unweighted_fit"

    try:
        model.fit(X, y, sample_weight=weights)
        return model, "native_sample_weight"
    except TypeError:
        # Compatibility fallback for estimators/versions without sample_weight.
        probabilities = np.clip(weights, 1e-12, None)
        probabilities = probabilities / probabilities.sum()
        rng = np.random.default_rng(random_state)
        indices = rng.choice(
            np.arange(len(X)),
            size=len(X),
            replace=True,
            p=probabilities,
        )
        model.fit(X.iloc[indices].reset_index(drop=True), np.asarray(y)[indices])
        return model, "weighted_bootstrap_fallback"


def predict_score(model, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X)[:, 1], dtype=float)
    if hasattr(model, "decision_function"):
        z = np.asarray(model.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
    return np.asarray(model.predict(X), dtype=float)


# ============================================================
# 7. Bias states: generation and exposure are separated
# ============================================================


@dataclass
class VectorState:
    stage: str
    vector: np.ndarray
    component_names: List[str]


@dataclass
class StageSnapshot:
    stage: str
    X: pd.DataFrame
    y: np.ndarray
    protected: np.ndarray
    weights: np.ndarray


def weighted_mean(values: np.ndarray, weights: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights)
    if not np.any(valid):
        return 0.0
    values = values[valid]
    weights = weights[valid]
    total = float(np.sum(weights))
    if total <= 0:
        return float(np.mean(values)) if len(values) else 0.0
    return float(np.sum(values * weights) / total)


def normalized_group_mean_gap(
    values: np.ndarray,
    protected: np.ndarray,
    weights: np.ndarray,
    eps: float = 1e-8,
) -> float:
    values = np.asarray(values, dtype=float)
    p = np.asarray(protected).astype(int)
    w = np.asarray(weights, dtype=float)

    valid = np.isfinite(values)
    if np.sum(valid & (p == 0)) == 0 or np.sum(valid & (p == 1)) == 0:
        return 0.0

    mean1 = weighted_mean(values[(p == 1) & valid], w[(p == 1) & valid])
    mean0 = weighted_mean(values[(p == 0) & valid], w[(p == 0) & valid])
    scale = float(np.nanstd(values[valid]) + eps)
    return float((mean1 - mean0) / scale)


def group_rate_gap(values: np.ndarray, protected: np.ndarray, weights: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    p = np.asarray(protected).astype(int)
    w = np.asarray(weights, dtype=float)
    if np.sum(p == 0) == 0 or np.sum(p == 1) == 0:
        return 0.0
    rate1 = weighted_mean(values[p == 1], w[p == 1])
    rate0 = weighted_mean(values[p == 0], w[p == 0])
    return float(rate1 - rate0)


def generation_state(snapshot: StageSnapshot, all_columns: Sequence[str]) -> VectorState:
    X = snapshot.X.reindex(columns=list(all_columns)).copy()
    y = np.asarray(snapshot.y).astype(int)
    p = np.asarray(snapshot.protected).astype(int)
    w = np.asarray(snapshot.weights, dtype=float)

    components: List[float] = []
    names: List[str] = []

    for feature in all_columns:
        values = pd.to_numeric(X[feature], errors="coerce").to_numpy(dtype=float)
        components.append(normalized_group_mean_gap(values, p, w))
        names.append(f"mean::{feature}")

    for feature in all_columns:
        missing = X[feature].isna().astype(float).to_numpy()
        components.append(group_rate_gap(missing, p, w))
        names.append(f"missing::{feature}")

    components.append(group_rate_gap(y.astype(float), p, w))
    names.append("label_gap")

    prevalence = float(np.mean(p)) if len(p) else 0.0
    components.append(prevalence - 0.5)
    names.append("protected_prevalence")

    if len(p) and np.sum(p == 0) > 0 and np.sum(p == 1) > 0:
        weight_gap = float(np.mean(w[p == 1]) - np.mean(w[p == 0]))
    else:
        weight_gap = 0.0
    components.append(weight_gap)
    names.append("weight_gap")

    return VectorState(
        stage=snapshot.stage,
        vector=np.nan_to_num(np.asarray(components, dtype=float)),
        component_names=names,
    )


def safe_group_mean(values: np.ndarray, mask: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    mask = np.asarray(mask, dtype=bool)
    if np.sum(mask) == 0:
        return 0.0
    return float(np.mean(values[mask]))


def score_bias_components(
    y_true: np.ndarray,
    scores: np.ndarray,
    protected: np.ndarray,
) -> Tuple[List[str], np.ndarray]:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    p = np.asarray(protected).astype(int)
    g1, g0 = p == 1, p == 0

    mean_gap = safe_group_mean(scores, g1) - safe_group_mean(scores, g0)
    brier1 = safe_group_mean((scores - y_true) ** 2, g1)
    brier0 = safe_group_mean((scores - y_true) ** 2, g0)
    brier_gap = brier1 - brier0

    positive_mask = y_true == 1
    negative_mask = y_true == 0
    pos_score_gap = (
        safe_group_mean(scores, g1 & positive_mask)
        - safe_group_mean(scores, g0 & positive_mask)
    )
    neg_score_gap = (
        safe_group_mean(scores, g1 & negative_mask)
        - safe_group_mean(scores, g0 & negative_mask)
    )

    names = [
        "score::mean_gap",
        "score::brier_gap",
        "score::positive_label_gap",
        "score::negative_label_gap",
        "decision::SPD",
        "decision::EOD",
        "decision::FPR_gap",
    ]
    vector = np.array([
        mean_gap,
        brier_gap,
        pos_score_gap,
        neg_score_gap,
        0.0,
        0.0,
        0.0,
    ], dtype=float)
    return names, vector


def decision_bias_components(
    y_true: np.ndarray,
    scores: np.ndarray,
    y_pred: np.ndarray,
    protected: np.ndarray,
) -> Tuple[List[str], np.ndarray]:
    names, vector = score_bias_components(y_true, scores, protected)
    metrics = compute_metrics(y_true, y_pred, protected, scores)
    vector[4] = metrics["SPD"]
    vector[5] = metrics["EOD"]
    vector[6] = metrics["FPR_gap"]
    return names, vector


def controlled_transition_report(
    faulty_states: Sequence[VectorState],
    reference_states: Sequence[VectorState],
    tau: float,
    channel: str,
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    if len(faulty_states) != len(reference_states):
        raise ValueError("Faulty and reference state sequences must have equal length")

    rows: List[Dict[str, Any]] = []
    generation_vectors: Dict[str, np.ndarray] = {}

    for index in range(1, len(faulty_states)):
        f_prev, f_curr = faulty_states[index - 1], faulty_states[index]
        r_prev, r_curr = reference_states[index - 1], reference_states[index]

        if f_curr.component_names != r_curr.component_names:
            raise ValueError(f"State component mismatch at stage {f_curr.stage}")

        generation_vector = (f_curr.vector - f_prev.vector) - (r_curr.vector - r_prev.vector)
        deviation_vector = f_curr.vector - r_curr.vector
        generation_norm = safe_norm(generation_vector)
        deviation_norm = safe_norm(deviation_vector)

        generation_vectors[f_curr.stage] = generation_vector
        rows.append({
            "channel": channel,
            "stage_index": index,
            "stage": f_curr.stage,
            "generation_norm": generation_norm,
            "transition_norm": generation_norm,
            "deviation_norm": deviation_norm,
            "flagged": bool(generation_norm >= float(tau)),
        })

    report = pd.DataFrame(rows)
    if len(report):
        report = report.sort_values(
            ["generation_norm", "stage_index"],
            ascending=[False, True],
        ).reset_index(drop=True)
        report["rank"] = np.arange(1, len(report) + 1)
        total = float(report["generation_norm"].sum())
        report["normalized_suspicion"] = (
            report["generation_norm"] / total if total > 0 else 0.0
        )
    return report, generation_vectors


# ============================================================
# 8. Metrics and ranking evaluation
# ============================================================


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    protected: np.ndarray,
    y_score: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    p = np.asarray(protected).astype(int)

    g1, g0 = p == 1, p == 0
    p1 = safe_group_mean(y_pred, g1)
    p0 = safe_group_mean(y_pred, g0)
    di = safe_div(p1, p0)
    spd = p1 - p0

    def tpr(group: int) -> float:
        mask = (p == group) & (y_true == 1)
        return safe_group_mean(y_pred, mask)

    def fpr(group: int) -> float:
        mask = (p == group) & (y_true == 0)
        return safe_group_mean(y_pred, mask)

    tpr1, tpr0 = tpr(1), tpr(0)
    fpr1, fpr0 = fpr(1), fpr(0)
    eod = tpr1 - tpr0
    fpr_gap = fpr1 - fpr0
    aod = 0.5 * (eod + fpr_gap)

    auc = np.nan
    if y_score is not None and len(np.unique(y_true)) == 2:
        try:
            auc = float(roc_auc_score(y_true, y_score))
        except Exception:
            auc = np.nan

    return {
        "DI": float(di),
        "|DI-1|": float(abs(di - 1.0)) if np.isfinite(di) else np.nan,
        "SPD": float(spd),
        "|SPD|": float(abs(spd)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc": auc,
        "EOD": float(eod),
        "|EOD|": float(abs(eod)),
        "FPR_gap": float(fpr_gap),
        "|FPR_gap|": float(abs(fpr_gap)),
        "AOD": float(aod),
        "|AOD|": float(abs(aod)),
    }


def ranking_metrics(
    predicted_ranking: Sequence[str],
    true_items: Sequence[str],
    k_values: Sequence[int],
    prefix: str,
) -> Dict[str, Any]:
    ranking = list(predicted_ranking)
    true_set = set(true_items)
    out: Dict[str, Any] = {}

    if not true_set:
        out[f"{prefix}_applicable"] = 0
        for k in k_values:
            out[f"{prefix}_precision@{k}"] = np.nan
            out[f"{prefix}_recall@{k}"] = np.nan
            out[f"{prefix}_all_hit@{k}"] = np.nan
        out[f"{prefix}_mrr"] = np.nan
        out[f"{prefix}_ndcg"] = np.nan
        return out

    out[f"{prefix}_applicable"] = 1
    for k in k_values:
        top = ranking[: int(k)]
        hits = len(true_set.intersection(top))
        out[f"{prefix}_precision@{k}"] = hits / max(len(top), 1)
        out[f"{prefix}_recall@{k}"] = hits / len(true_set)
        out[f"{prefix}_all_hit@{k}"] = int(true_set.issubset(set(top)))

    reciprocal_ranks = [
        1.0 / (ranking.index(item) + 1)
        for item in true_set if item in ranking
    ]
    out[f"{prefix}_mrr"] = max(reciprocal_ranks) if reciprocal_ranks else 0.0

    relevance = np.array([1.0 if item in true_set else 0.0 for item in ranking], dtype=float)
    if len(relevance) == 0:
        ndcg = 0.0
    else:
        discounts = 1.0 / np.log2(np.arange(2, len(relevance) + 2))
        dcg = float(np.sum(relevance * discounts))
        ideal = np.sort(relevance)[::-1]
        idcg = float(np.sum(ideal * discounts))
        ndcg = dcg / idcg if idcg > 0 else 0.0
    out[f"{prefix}_ndcg"] = ndcg
    return out


# ============================================================
# 9. RCA: fault delta is separated from ordinary proxy MI
# ============================================================


def proxy_mi_scores(X: pd.DataFrame, protected: np.ndarray, seed: int) -> Dict[str, float]:
    Xn = X.copy()
    Xn = Xn.apply(pd.to_numeric, errors="coerce")
    Xn = Xn.replace([np.inf, -np.inf], np.nan).fillna(0)
    p = np.asarray(protected).astype(int)
    if len(np.unique(p)) < 2 or Xn.shape[1] == 0:
        return {col: 0.0 for col in Xn.columns}
    scores = mutual_info_classif(
        Xn.to_numpy(dtype=float),
        p,
        random_state=seed,
        discrete_features=False,
    )
    return dict(zip(Xn.columns, scores.astype(float)))


def component_to_feature(component: str) -> Optional[str]:
    if component.startswith("mean::"):
        return component.split("::", 1)[1]
    if component.startswith("missing::"):
        return component.split("::", 1)[1]
    return None


def make_rca_table(
    stage: str,
    generation_vector: np.ndarray,
    component_names: Sequence[str],
    faulty_snapshot: Optional[StageSnapshot],
    seed: int,
) -> pd.DataFrame:
    if len(generation_vector) != len(component_names):
        raise ValueError("RCA vector and component names differ in length")

    mi_map: Dict[str, float] = {}
    if faulty_snapshot is not None:
        mi_map = proxy_mi_scores(faulty_snapshot.X, faulty_snapshot.protected, seed)

    rows: List[Dict[str, Any]] = []
    for component, value in zip(component_names, generation_vector):
        feature = component_to_feature(component)
        rows.append({
            "stage": stage,
            "component": component,
            "component_type": component.split("::", 1)[0] if "::" in component else component,
            "feature": feature,
            "fault_delta_score": float(abs(value)),
            "signed_fault_delta": float(value),
            "proxy_mi_score": float(mi_map.get(feature, np.nan)) if feature else np.nan,
        })

    df = pd.DataFrame(rows).sort_values(
        ["fault_delta_score", "proxy_mi_score"],
        ascending=[False, False],
        na_position="last",
    ).reset_index(drop=True)
    df["rank"] = np.arange(1, len(df) + 1)
    return df


# ============================================================
# 10. One reference/faulty pipeline run
# ============================================================


@dataclass
class PipelineArtifacts:
    generation_states: List[VectorState]
    generation_snapshots: Dict[str, StageSnapshot]
    exposure_states: List[VectorState]
    scores: np.ndarray
    predictions: np.ndarray
    metrics: Dict[str, float]
    model_fit_mode: str
    fault_log: Dict[str, Any]


def run_pipeline_variant(
    X_train_raw: pd.DataFrame,
    y_train_raw: np.ndarray,
    p_train_raw: np.ndarray,
    X_test_raw: pd.DataFrame,
    y_test: np.ndarray,
    p_test: np.ndarray,
    scenario: FaultScenario,
    model_name: str,
    seed: int,
    reference_mode: bool,
    all_columns: Sequence[str],
) -> PipelineArtifacts:
    active = FaultScenario(
        scenario_id="REFERENCE",
        description="No faults",
        faults=tuple(),
    ) if reference_mode else scenario

    base_seed = stable_int_seed(seed, model_name, scenario.scenario_id, "reference" if reference_mode else "faulty")

    X0 = X_train_raw.copy().reset_index(drop=True)
    y0 = np.asarray(y_train_raw).astype(int).copy()
    p0 = np.asarray(p_train_raw).astype(int).copy()
    ones0 = np.ones(len(y0), dtype=float)

    snapshots: Dict[str, StageSnapshot] = {
        "raw_train": StageSnapshot("raw_train", X0.copy(), y0.copy(), p0.copy(), ones0.copy())
    }
    fault_log: Dict[str, Any] = {}

    # S1: acquisition/selection faults affect training data only.
    rng_acq = np.random.default_rng(stable_int_seed(base_seed, "acquisition"))
    X1, y1, p1, acquisition_log = apply_acquisition_faults(
        X0,
        y0,
        p0,
        faults_at(active, "data_acquisition"),
        rng_acq,
    )
    w1 = np.ones(len(y1), dtype=float)
    snapshots["data_acquisition"] = StageSnapshot(
        "data_acquisition", X1.copy(), y1.copy(), p1.copy(), w1.copy()
    )
    fault_log["data_acquisition"] = acquisition_log

    # S2a: cleaning faults apply to train and test pipeline inputs.
    rng_clean_train = np.random.default_rng(stable_int_seed(base_seed, "clean_train"))
    rng_clean_test = np.random.default_rng(stable_int_seed(base_seed, "clean_test"))
    X2, cleaning_train_log = apply_cleaning_faults(
        X1,
        p1,
        faults_at(active, "data_cleaning"),
        rng_clean_train,
    )
    X_test_clean, cleaning_test_log = apply_cleaning_faults(
        X_test_raw.copy().reset_index(drop=True),
        p_test,
        faults_at(active, "data_cleaning"),
        rng_clean_test,
    )
    snapshots["data_cleaning"] = StageSnapshot(
        "data_cleaning", X2.copy(), y1.copy(), p1.copy(), w1.copy()
    )
    fault_log["data_cleaning_train"] = cleaning_train_log
    fault_log["data_cleaning_test"] = cleaning_test_log

    # S2b: label-processing faults affect training labels only.
    rng_label = np.random.default_rng(stable_int_seed(base_seed, "label"))
    y3, label_log = apply_label_faults(
        y1,
        p1,
        faults_at(active, "label_processing"),
        rng_label,
    )
    snapshots["label_processing"] = StageSnapshot(
        "label_processing", X2.copy(), y3.copy(), p1.copy(), w1.copy()
    )
    fault_log["label_processing"] = label_log

    # Imputation is part of the stage transition into feature engineering.
    imputer = SimpleImputer(strategy="median")
    X2_imp = pd.DataFrame(
        imputer.fit_transform(X2),
        columns=list(all_columns),
    )
    Xtest_imp = pd.DataFrame(
        imputer.transform(X_test_clean.reindex(columns=list(all_columns))),
        columns=list(all_columns),
    )

    # S3: group-specific feature transformation.
    feature_transformer = FeatureFaultTransformer(
        faults_at(active, "feature_engineering")
    ).fit(X2_imp)
    X4 = feature_transformer.transform(X2_imp, p1)
    Xtest4 = feature_transformer.transform(Xtest_imp, p_test)
    snapshots["feature_engineering"] = StageSnapshot(
        "feature_engineering", X4.copy(), y3.copy(), p1.copy(), w1.copy()
    )

    # Standardize final model matrix.
    scaler = StandardScaler()
    X4_scaled = pd.DataFrame(
        scaler.fit_transform(X4),
        columns=list(all_columns),
    )
    Xtest_scaled = pd.DataFrame(
        scaler.transform(Xtest4),
        columns=list(all_columns),
    )

    # S4: effective training distribution through sample weighting.
    weights, weight_log = build_sample_weights(
        y3,
        p1,
        faults_at(active, "training_effective"),
    )
    snapshots["training_effective"] = StageSnapshot(
        "training_effective",
        X4_scaled.copy(),
        y3.copy(),
        p1.copy(),
        weights.copy(),
    )
    fault_log["training_effective"] = weight_log

    model = make_model(model_name, random_state=seed)
    model, fit_mode = fit_model_with_weights(
        model,
        X4_scaled,
        y3,
        weights,
        random_state=stable_int_seed(seed, scenario.scenario_id, model_name, "weighted_fit"),
    )

    base_scores = predict_score(model, Xtest_scaled)
    scores = apply_score_faults(
        base_scores,
        p_test,
        faults_at(active, "model_score"),
    )
    predictions = apply_threshold_faults(
        scores,
        p_test,
        faults_at(active, "decision_threshold"),
        default_threshold=0.5,
    )

    generation_states = [
        generation_state(snapshots[stage], all_columns)
        for stage in GENERATION_STAGES
    ]

    score_names, score_vector = score_bias_components(y_test, scores, p_test)
    decision_names, decision_vector = decision_bias_components(
        y_test,
        scores,
        predictions,
        p_test,
    )
    exposure_states = [
        VectorState("pre_output", np.zeros(len(score_names), dtype=float), score_names),
        VectorState("model_score", score_vector, score_names),
        VectorState("decision_threshold", decision_vector, decision_names),
    ]

    return PipelineArtifacts(
        generation_states=generation_states,
        generation_snapshots=snapshots,
        exposure_states=exposure_states,
        scores=scores,
        predictions=predictions,
        metrics=compute_metrics(y_test, predictions, p_test, scores),
        model_fit_mode=fit_mode,
        fault_log=fault_log,
    )


def top_stage(report: pd.DataFrame, require_flag: bool = False) -> Optional[str]:
    if report is None or len(report) == 0:
        return None
    if require_flag and not bool(report.iloc[0]["flagged"]):
        return None
    return str(report.iloc[0]["stage"])


def earliest_flagged_stage(report: pd.DataFrame) -> Optional[str]:
    if report is None or len(report) == 0:
        return None
    flagged = report[report["flagged"]].copy()
    if len(flagged) == 0:
        return None
    flagged["_order"] = flagged["stage"].map(STAGE_ORDER)
    return str(flagged.sort_values("_order").iloc[0]["stage"])


def near_max_stages(
    report: pd.DataFrame,
    ratio: float = 0.99,
    require_flag: bool = True,
) -> List[str]:
    """Return stages whose transition norm is nearly tied with the maximum.

    A cleaning fault can be introduced at data_cleaning and then produce a nearly
    identical reverse/echo transition when the imputer consumes the missing values.
    The near-maximum set lets us identify the earliest root rather than relabeling
    the downstream processing echo as the original fault stage.
    """
    if report is None or len(report) == 0:
        return []
    work = report.copy()
    max_norm = float(work["generation_norm"].max())
    if not np.isfinite(max_norm) or max_norm <= 0.0:
        return []
    threshold = float(np.clip(ratio, 0.0, 1.0)) * max_norm
    candidates = work[work["generation_norm"] >= threshold].copy()
    if require_flag:
        candidates = candidates[candidates["flagged"]]
    if len(candidates) == 0:
        return []
    candidates["_order"] = candidates["stage"].map(STAGE_ORDER)
    return candidates.sort_values(["_order", "generation_norm"], ascending=[True, False])["stage"].astype(str).tolist()


def earliest_near_max_stage(
    report: pd.DataFrame,
    ratio: float = 0.99,
    require_flag: bool = True,
) -> Optional[str]:
    stages = near_max_stages(report, ratio=ratio, require_flag=require_flag)
    return stages[0] if stages else None


def root_aware_stage_ranking(report: pd.DataFrame, ratio: float = 0.99) -> List[str]:
    """Re-rank near-tied maxima by causal stage order, retaining all other ranks."""
    if report is None or len(report) == 0:
        return []
    raw = report["stage"].astype(str).tolist()
    roots = near_max_stages(report, ratio=ratio, require_flag=True)
    if not roots:
        return raw
    root_set = set(roots)
    return roots + [stage for stage in raw if stage not in root_set]


def annotate_root_and_echoes(report: pd.DataFrame, ratio: float) -> pd.DataFrame:
    out = report.copy()
    roots = near_max_stages(out, ratio=ratio, require_flag=True)
    root = roots[0] if roots else None
    root_order = STAGE_ORDER.get(root, -1) if root is not None else -1
    near_set = set(roots)
    out["near_max"] = out["stage"].astype(str).isin(near_set)
    out["root_candidate"] = out["stage"].astype(str).eq(root) if root is not None else False
    out["downstream_echo"] = out.apply(
        lambda row: bool(
            row["stage"] in near_set
            and root is not None
            and STAGE_ORDER.get(str(row["stage"]), -1) > root_order
        ),
        axis=1,
    )
    out["root_stage"] = root
    out["near_max_ratio"] = float(ratio)
    return out


def run_one_experiment(
    X: pd.DataFrame,
    y: np.ndarray,
    protected: np.ndarray,
    scenario: FaultScenario,
    model_name: str,
    seed: int,
    config: Dict[str, Any],
) -> Dict[str, pd.DataFrame]:
    wall_start = time.perf_counter()
    cpu_start = time.process_time()

    X_train, X_test, y_train, y_test, p_train, p_test = train_test_split(
        X,
        y,
        protected,
        test_size=float(config["TEST_SIZE"]),
        random_state=seed,
        stratify=y,
    )
    X_train = pd.DataFrame(X_train, columns=X.columns).reset_index(drop=True)
    X_test = pd.DataFrame(X_test, columns=X.columns).reset_index(drop=True)
    y_train = np.asarray(y_train).astype(int)
    y_test = np.asarray(y_test).astype(int)
    p_train = np.asarray(p_train).astype(int)
    p_test = np.asarray(p_test).astype(int)

    reference = run_pipeline_variant(
        X_train,
        y_train,
        p_train,
        X_test,
        y_test,
        p_test,
        scenario,
        model_name,
        seed,
        reference_mode=True,
        all_columns=X.columns,
    )
    faulty = run_pipeline_variant(
        X_train,
        y_train,
        p_train,
        X_test,
        y_test,
        p_test,
        scenario,
        model_name,
        seed,
        reference_mode=False,
        all_columns=X.columns,
    )

    generation_report, generation_vectors = controlled_transition_report(
        faulty.generation_states,
        reference.generation_states,
        tau=float(config["GENERATION_TAU"]),
        channel="generation",
    )
    exposure_report, exposure_vectors = controlled_transition_report(
        faulty.exposure_states,
        reference.exposure_states,
        tau=float(config["EXPOSURE_TAU"]),
        channel="manifestation",
    )

    near_max_ratio = float(config.get("ROOT_NEAR_MAX_RATIO", 0.99))
    generation_report = annotate_root_and_echoes(generation_report, near_max_ratio)
    exposure_report = exposure_report.copy()
    exposure_report["manifestation_norm"] = exposure_report["generation_norm"]

    # Add run metadata to reports.
    common_meta = {
        "scenario_id": scenario.scenario_id,
        "scenario_category": scenario.category,
        "strength": scenario.strength,
        "model_name": model_name,
        "seed": seed,
    }
    for key, value in common_meta.items():
        generation_report[key] = value
        exposure_report[key] = value

    # Fault-delta RCA is cheap and is computed for every stage. Ordinary protected-proxy
    # MI is deliberately computed only once at the highest-ranked generation stage;
    # recomputing MI for every stage makes a long campaign unnecessarily expensive.
    rca_frames: List[pd.DataFrame] = []
    component_names = faulty.generation_states[0].component_names
    for stage, vector in generation_vectors.items():
        table = make_rca_table(stage, vector, component_names, None, seed)
        for key, value in common_meta.items():
            table[key] = value
        table["channel"] = "generation"
        rca_frames.append(table)

    exposure_names = faulty.exposure_states[0].component_names
    for stage, vector in exposure_vectors.items():
        table = make_rca_table(stage, vector, exposure_names, None, seed)
        for key, value in common_meta.items():
            table[key] = value
        table["channel"] = "exposure"
        rca_frames.append(table)

    rca_full = pd.concat(rca_frames, ignore_index=True) if rca_frames else pd.DataFrame()

    primary_carriers = set(scenario.primary_carriers)
    secondary_carriers = set(scenario.secondary_carriers)
    if len(rca_full):
        rca_full["is_primary_carrier"] = rca_full["component"].isin(primary_carriers)
        rca_full["is_secondary_carrier"] = rca_full["component"].isin(secondary_carriers)
        rca_full["carrier_role"] = np.select(
            [rca_full["is_primary_carrier"], rca_full["is_secondary_carrier"]],
            ["primary", "secondary"],
            default="none",
        )

    generation_raw_ranking = generation_report["stage"].astype(str).tolist()
    generation_ranking = root_aware_stage_ranking(generation_report, near_max_ratio)
    manifestation_ranking = exposure_report["stage"].astype(str).tolist()
    # Backward-compatible name for prior analysis scripts.
    exposure_ranking = manifestation_ranking

    # RCA ranking is aggregated across the true stages when available. For a decision-only
    # scenario, use exposure components. For control, use the top channel/stage.
    relevant_rca_rows: List[pd.DataFrame] = []
    for stage in scenario.true_generation_stages:
        relevant_rca_rows.append(rca_full[(rca_full["channel"] == "generation") & (rca_full["stage"] == stage)])
    for stage in scenario.true_exposure_stages:
        relevant_rca_rows.append(rca_full[(rca_full["channel"] == "exposure") & (rca_full["stage"] == stage)])

    if relevant_rca_rows:
        relevant_rca = pd.concat(relevant_rca_rows, ignore_index=True)
        rca_aggregate = (
            relevant_rca.groupby("component", as_index=False)["fault_delta_score"]
            .max()
            .sort_values("fault_delta_score", ascending=False)
        )
        rca_ranking = rca_aggregate["component"].tolist()
    else:
        rca_ranking = []

    # Add ordinary proxy MI only at the highest-ranked generation stage. It is a
    # descriptive proxy signal, not the controlled fault-delta RCA used for accuracy.
    if bool(config.get("COMPUTE_PROXY_MI", True)) and generation_ranking:
        mi_stage = generation_ranking[0]
        mi_snapshot = faulty.generation_snapshots.get(mi_stage)
        if mi_snapshot is not None:
            mi_map = proxy_mi_scores(mi_snapshot.X, mi_snapshot.protected, seed)
            mask = (rca_full["channel"] == "generation") & (rca_full["stage"] == mi_stage)
            rca_full.loc[mask, "proxy_mi_score"] = rca_full.loc[mask, "feature"].map(mi_map)

    save_top_n = int(config.get("RCA_SAVE_TOP_N", 50))
    if len(rca_full):
        rca = (
            rca_full.sort_values(["channel", "stage", "rank"])
            .groupby(["channel", "stage"], as_index=False, group_keys=False)
            .head(save_top_n)
            .reset_index(drop=True)
        )
    else:
        rca = rca_full.copy()

    generation_root_stage = earliest_near_max_stage(
        generation_report, ratio=near_max_ratio, require_flag=True
    )
    generation_near_max = near_max_stages(
        generation_report, ratio=near_max_ratio, require_flag=True
    )
    root_order = STAGE_ORDER.get(generation_root_stage, -1) if generation_root_stage else -1
    downstream_echoes = [
        stage for stage in generation_near_max
        if STAGE_ORDER.get(stage, -1) > root_order
    ]
    manifestation_top1 = top_stage(exposure_report, require_flag=True)

    result: Dict[str, Any] = {
        **common_meta,
        "schema_version": SCHEMA_VERSION,
        "description": scenario.description,
        "expected_fault_role": scenario.expected_fault_role,
        "fault_count": len(scenario.faults),
        "faults_json": json_dumps_safe([f.as_dict() for f in scenario.faults]),
        "true_generation_stages": ", ".join(scenario.true_generation_stages),
        "true_exposure_stages": ", ".join(scenario.true_exposure_stages),
        "true_primary_carriers": ", ".join(scenario.primary_carriers),
        "true_secondary_carriers": ", ".join(scenario.secondary_carriers),
        # Backward-compatible alias: true_carriers now means direct/primary causes.
        "true_carriers": ", ".join(scenario.primary_carriers),

        # Raw magnitude rank and root-aware localization are deliberately separated.
        "generation_rank1_raw": top_stage(generation_report, require_flag=False),
        "generation_top1_raw": top_stage(generation_report, require_flag=True),
        "generation_root_stage": generation_root_stage,
        "generation_top1": generation_root_stage,
        "generation_first_flagged": earliest_flagged_stage(generation_report),
        "generation_near_max_stages": ", ".join(generation_near_max),
        "generation_downstream_echo_stages": ", ".join(downstream_echoes),
        "generation_max_norm": float(generation_report["generation_norm"].max()) if len(generation_report) else 0.0,
        "generation_root_hit": (
            int(generation_root_stage in set(scenario.true_generation_stages))
            if scenario.true_generation_stages and generation_root_stage is not None else np.nan
        ),
        "generation_first_flagged_hit": (
            int(earliest_flagged_stage(generation_report) in set(scenario.true_generation_stages))
            if scenario.true_generation_stages else np.nan
        ),

        # Output-stage changes are manifestations. Only score/threshold scenarios have
        # a true injected exposure stage.
        "manifestation_rank1": top_stage(exposure_report, require_flag=False),
        "manifestation_top1": manifestation_top1,
        "manifestation_first_flagged": earliest_flagged_stage(exposure_report),
        "manifestation_max_norm": float(exposure_report["generation_norm"].max()) if len(exposure_report) else 0.0,
        "injected_exposure_stages": ", ".join(scenario.true_exposure_stages),
        "injected_exposure_hit@1": (
            int(manifestation_top1 in set(scenario.true_exposure_stages))
            if scenario.true_exposure_stages and manifestation_top1 is not None else np.nan
        ),
        # Backward-compatible exposure aliases.
        "exposure_rank1": top_stage(exposure_report, require_flag=False),
        "exposure_top1": manifestation_top1,
        "exposure_first_flagged": earliest_flagged_stage(exposure_report),
        "exposure_max_norm": float(exposure_report["generation_norm"].max()) if len(exposure_report) else 0.0,

        "rca_top1": rca_ranking[0] if rca_ranking else None,
        "rca_primary_top1_hit": (
            int(bool(rca_ranking) and rca_ranking[0] in primary_carriers)
            if primary_carriers else np.nan
        ),
        "rca_secondary_top1_hit": (
            int(bool(rca_ranking) and rca_ranking[0] in secondary_carriers)
            if secondary_carriers else np.nan
        ),
        "model_fit_mode": faulty.model_fit_mode,
        "reference_model_fit_mode": reference.model_fit_mode,
        "fault_log_json": json_dumps_safe(faulty.fault_log),
    }
    result.update(ranking_metrics(
        generation_ranking,
        scenario.true_generation_stages,
        config["EVAL_K_VALUES"],
        prefix="generation",
    ))
    result.update(ranking_metrics(
        generation_raw_ranking,
        scenario.true_generation_stages,
        config["EVAL_K_VALUES"],
        prefix="generation_raw",
    ))
    result.update(ranking_metrics(
        manifestation_ranking,
        scenario.true_exposure_stages,
        config["EVAL_K_VALUES"],
        prefix="injected_exposure",
    ))
    # Backward-compatible exposure metric aliases.
    result.update(ranking_metrics(
        exposure_ranking,
        scenario.true_exposure_stages,
        config["EVAL_K_VALUES"],
        prefix="exposure",
    ))
    result.update(ranking_metrics(
        rca_ranking,
        scenario.primary_carriers,
        config["EVAL_K_VALUES"],
        prefix="rca",
    ))
    result.update(ranking_metrics(
        rca_ranking,
        scenario.secondary_carriers,
        config["EVAL_K_VALUES"],
        prefix="rca_secondary",
    ))

    # Save both faulty metrics and reference deltas.
    for metric, value in faulty.metrics.items():
        result[metric] = value
        result[f"faulty_{metric}"] = value
    for metric, reference_value in reference.metrics.items():
        result[f"reference_{metric}"] = reference_value
        faulty_value = faulty.metrics.get(metric, np.nan)
        result[f"delta_{metric}"] = faulty_value - reference_value

    fairness_effect_deltas = {
        "SPD": float(faulty.metrics["|SPD|"] - reference.metrics["|SPD|"]),
        "AOD": float(faulty.metrics["|AOD|"] - reference.metrics["|AOD|"]),
        "EOD": float(faulty.metrics["|EOD|"] - reference.metrics["|EOD|"]),
    }
    fairness_harm_deltas = np.asarray(list(fairness_effect_deltas.values()), dtype=float)
    fairness_harm_mean = float(np.nanmean(fairness_harm_deltas))
    effect_eps = float(config.get("FAIRNESS_EFFECT_EPS", 1e-4))

    positive = [name for name, value in fairness_effect_deltas.items() if value > effect_eps]
    negative = [name for name, value in fairness_effect_deltas.items() if value < -effect_eps]
    neutral = [
        name for name, value in fairness_effect_deltas.items()
        if abs(value) <= effect_eps
    ]
    if len(positive) == len(fairness_effect_deltas):
        observed_effect = "amplifying"
    elif len(negative) == len(fairness_effect_deltas):
        observed_effect = "compensating"
    elif len(neutral) == len(fairness_effect_deltas):
        observed_effect = "neutral_or_latent"
    else:
        # Fairness notions can move in opposite directions; do not collapse this
        # trade-off into a misleading single "improved" or "worsened" label.
        observed_effect = "mixed_metric_tradeoff"

    result["fairness_harm_delta_mean"] = fairness_harm_mean
    result["fairness_metrics_amplified"] = ", ".join(positive)
    result["fairness_metrics_compensated"] = ", ".join(negative)
    result["fairness_metrics_neutral"] = ", ".join(neutral)
    result["observed_fairness_effect"] = observed_effect
    result["expected_effect_matches_observed"] = (
        int(scenario.expected_fault_role == observed_effect)
        if scenario.expected_fault_role in {"amplifying", "compensating"}
        and observed_effect in {"amplifying", "compensating"}
        else np.nan
    )

    wall_seconds = float(time.perf_counter() - wall_start)
    cpu_seconds = float(time.process_time() - cpu_start)
    result["runtime_wall_seconds"] = wall_seconds
    result["runtime_cpu_seconds"] = cpu_seconds
    # Backward-compatible alias.
    result["runtime_seconds"] = wall_seconds

    return {
        "results": pd.DataFrame([result]),
        "generation_report": generation_report,
        "exposure_report": exposure_report,
        "rca": rca,
    }


# ============================================================
# 11. Campaign execution, checkpointing, and summaries
# ============================================================


def output_paths(prefix: str) -> Dict[str, Path]:
    return {
        "results": Path(f"{prefix}_results.csv"),
        "generation": Path(f"{prefix}_generation_reports.csv"),
        "exposure": Path(f"{prefix}_exposure_reports.csv"),
        "rca": Path(f"{prefix}_rca.csv"),
        "summary": Path(f"{prefix}_summary.csv"),
        "paper": Path(f"{prefix}_paper_table.csv"),
        "scenarios": Path(f"{prefix}_scenarios.json"),
    }


def reset_outputs(paths: Dict[str, Path]) -> None:
    for key, path in paths.items():
        if key == "scenarios":
            continue
        if path.exists():
            path.unlink()


def load_completed_keys(results_path: Path) -> set[Tuple[str, str, int]]:
    if not results_path.exists() or results_path.stat().st_size == 0:
        return set()
    existing = pd.read_csv(results_path)
    required = {"scenario_id", "model_name", "seed"}
    if not required.issubset(existing.columns):
        return set()
    return {
        (str(row.scenario_id), str(row.model_name), int(row.seed))
        for row in existing[["scenario_id", "model_name", "seed"]].itertuples(index=False)
    }


def build_summary(results: pd.DataFrame) -> pd.DataFrame:
    if len(results) == 0:
        return pd.DataFrame()

    identity_cols = [
        "scenario_id",
        "scenario_category",
        "strength",
        "model_name",
        "fault_count",
        "true_generation_stages",
        "true_exposure_stages",
        "true_primary_carriers",
        "true_secondary_carriers",
        "true_carriers",
        "expected_fault_role",
    ]
    candidate_metrics = [
        col for col in results.columns
        if col.startswith(("generation_", "manifestation_", "injected_exposure_", "exposure_", "rca_", "delta_"))
        or col in {
            "|DI-1|", "|SPD|", "accuracy", "recall", "f1", "auc",
            "|AOD|", "|EOD|", "|FPR_gap|",
            "fairness_harm_delta_mean", "expected_effect_matches_observed",
            "runtime_seconds", "runtime_wall_seconds", "runtime_cpu_seconds",
        }
    ]
    numeric_metrics = [
        col for col in candidate_metrics
        if col in results.columns and pd.api.types.is_numeric_dtype(results[col])
    ]

    grouped = results.groupby(identity_cols, dropna=False)
    summary = grouped[numeric_metrics].agg(["mean", "std"]).reset_index()
    summary = flatten_columns(summary)

    runtime_cols = [
        col for col in ["runtime_seconds", "runtime_wall_seconds", "runtime_cpu_seconds"]
        if col in results.columns and pd.api.types.is_numeric_dtype(results[col])
    ]
    if runtime_cols:
        runtime_median = grouped[runtime_cols].median().reset_index()
        runtime_median = runtime_median.rename(
            columns={col: f"{col}_median" for col in runtime_cols}
        )
        summary = summary.merge(runtime_median, on=identity_cols, how="left")
    return summary


def build_paper_table(summary: pd.DataFrame) -> pd.DataFrame:
    if len(summary) == 0:
        return pd.DataFrame()

    preferred = [
        "scenario_id",
        "scenario_category",
        "strength",
        "model_name",
        "fault_count",
        "expected_fault_role",
        "true_generation_stages",
        "true_exposure_stages",
        "true_primary_carriers",
        "true_secondary_carriers",
        "generation_root_hit_mean",
        "generation_first_flagged_hit_mean",
        "generation_recall@1_mean",
        "generation_recall@2_mean",
        "generation_recall@3_mean",
        "generation_raw_recall@1_mean",
        "generation_ndcg_mean",
        "injected_exposure_hit@1_mean",
        "injected_exposure_recall@1_mean",
        "injected_exposure_recall@2_mean",
        "rca_primary_top1_hit_mean",
        "rca_recall@1_mean",
        "rca_recall@3_mean",
        "rca_ndcg_mean",
        "rca_secondary_recall@3_mean",
        "fairness_harm_delta_mean_mean",
        "expected_effect_matches_observed_mean",
        "delta_|DI-1|_mean",
        "delta_|SPD|_mean",
        "delta_accuracy_mean",
        "delta_recall_mean",
        "delta_|AOD|_mean",
        "delta_|EOD|_mean",
        "runtime_wall_seconds_median",
        "runtime_cpu_seconds_median",
    ]
    return summary[[c for c in preferred if c in summary.columns]].copy()


def print_config(config: Dict[str, Any]) -> None:
    print("Using CONFIG:")
    for key, value in config.items():
        print(f"  {key}: {value}")


def main(config: Optional[Dict[str, Any]] = None) -> None:
    cfg = dict(CONFIG if config is None else config)
    print_config(cfg)

    paths = output_paths(cfg["OUTPUT_PREFIX"])
    for path in paths.values():
        path.parent.mkdir(parents=True, exist_ok=True)

    if bool(cfg.get("RESET_OUTPUTS", False)):
        reset_outputs(paths)

    X, y, protected, _, _ = load_compas_recidivism(
        cfg["CSV_PATH"],
        protected_positive_value=cfg["PROTECTED_POSITIVE_VALUE"],
        exclude_protected_from_model=not bool(cfg["INCLUDE_PROTECTED_FEATURE"]),
    )

    max_rows = cfg.get("MAX_ROWS")
    if max_rows is not None and int(max_rows) < len(X):
        # Stratified deterministic sample for smoke testing.
        original_columns = list(X.columns)
        X_sub, _, y_sub, _, p_sub, _ = train_test_split(
            X,
            y,
            protected,
            train_size=int(max_rows),
            random_state=0,
            stratify=y,
        )
        X = pd.DataFrame(X_sub, columns=original_columns).reset_index(drop=True)
        y = np.asarray(y_sub).astype(int)
        protected = np.asarray(p_sub).astype(int)
        print(f"Smoke-test subsample: {len(X)} rows")

    scenarios = build_campaign(cfg["CAMPAIGN_MODE"])
    selected_ids = cfg.get("SCENARIO_IDS")
    if selected_ids:
        selected_set = {str(x) for x in selected_ids}
        scenarios = [s for s in scenarios if s.scenario_id in selected_set]
        missing_ids = selected_set.difference({s.scenario_id for s in scenarios})
        if missing_ids:
            raise ValueError(f"Unknown SCENARIO_IDS: {sorted(missing_ids)}")
    if cfg.get("MAX_SCENARIOS") is not None:
        scenarios = scenarios[: int(cfg["MAX_SCENARIOS"])]

    with open(paths["scenarios"], "w", encoding="utf-8") as fp:
        json.dump([s.as_dict() for s in scenarios], fp, ensure_ascii=False, indent=2)

    completed = (
        load_completed_keys(paths["results"])
        if bool(cfg.get("RESUME", True)) else set()
    )

    total = len(scenarios) * len(cfg["MODELS"]) * int(cfg["REPEATS"])
    done_before = len(completed)
    print(f"\nCampaign runs: {total}; already completed: {done_before}")

    run_index = 0
    for scenario in scenarios:
        for model_name in cfg["MODELS"]:
            for repeat in range(int(cfg["REPEATS"])):
                seed = int(cfg["SEED_START"]) + repeat
                key = (scenario.scenario_id, str(model_name), seed)
                run_index += 1

                if key in completed:
                    print(f"[{run_index}/{total}] SKIP completed: {key}")
                    continue

                print(
                    f"\n[{run_index}/{total}] scenario={scenario.scenario_id} "
                    f"model={model_name} seed={seed}"
                )
                outputs = run_one_experiment(
                    X,
                    y,
                    protected,
                    scenario,
                    model_name,
                    seed,
                    cfg,
                )

                append_csv(outputs["results"], paths["results"])
                append_csv(outputs["generation_report"], paths["generation"])
                append_csv(outputs["exposure_report"], paths["exposure"])
                append_csv(outputs["rca"], paths["rca"])

                row = outputs["results"].iloc[0]
                print(
                    "  generation_root=", row.get("generation_root_stage"),
                    " raw_rank1=", row.get("generation_rank1_raw"),
                    " manifestation_top1=", row.get("manifestation_top1"),
                    " rca_top1=", row.get("rca_top1"),
                    " wall=", f"{row.get('runtime_wall_seconds', 0):.1f}s",
                    " cpu=", f"{row.get('runtime_cpu_seconds', 0):.1f}s",
                )

                if bool(cfg.get("VERBOSE_FIRST_RUN", True)) and run_index == 1:
                    print("\nGeneration report")
                    display(outputs["generation_report"])
                    print("\nOutput manifestation report")
                    display(outputs["exposure_report"])
                    print("\nTop RCA components")
                    display(outputs["rca"].sort_values("fault_delta_score", ascending=False).head(20))

    if paths["results"].exists():
        results = pd.read_csv(paths["results"])
        summary = build_summary(results)
        paper = build_paper_table(summary)
        summary.to_csv(paths["summary"], index=False, encoding="utf-8-sig")
        paper.to_csv(paths["paper"], index=False, encoding="utf-8-sig")

        print("\n================ CAMPAIGN COMPLETE ================")
        print("Results:", paths["results"])
        print("Generation reports:", paths["generation"])
        print("Exposure reports:", paths["exposure"])
        print("RCA:", paths["rca"])
        print("Summary:", paths["summary"])
        print("Paper table:", paths["paper"])
        print("Scenario manifest:", paths["scenarios"])
        print("Total completed rows:", len(results))
    else:
        print("No result file was created.")


if __name__ == "__main__":
    main()


Using CONFIG:
  CSV_PATH: /content/drive/MyDrive/Paper_Bias/LTDD/LTDD-main/datasets/compas-scores-two-years.csv
  OUTPUT_PREFIX: /content/drive/MyDrive/Paper_Bias/results_compas_full/biasflow_v4_1_compas_fault_campaign
  PROTECTED_POSITIVE_VALUE: African-American
  INCLUDE_PROTECTED_FEATURE: False
  TEST_SIZE: 0.3
  CAMPAIGN_MODE: full
  MODELS: ['lr', 'rf', 'hgb', 'mlp']
  REPEATS: 5
  SEED_START: 0
  SCENARIO_IDS: None
  EVAL_K_VALUES: [1, 2, 3, 5]
  GENERATION_TAU: 0.01
  EXPOSURE_TAU: 0.005
  ROOT_NEAR_MAX_RATIO: 0.95
  FAIRNESS_EFFECT_EPS: 0.0001
  RESUME: True
  RESET_OUTPUTS: False
  VERBOSE_FIRST_RUN: True
  COMPUTE_PROXY_MI: True
  RCA_SAVE_TOP_N: 50
  MAX_ROWS: None
  MAX_SCENARIOS: None
Loaded local CSV: /content/drive/MyDrive/Paper_Bias/LTDD/LTDD-main/datasets/compas-scores-two-years.csv

================ DATASET INFO ================
Dataset: COMPAS Recidivism
Target column: two_year_recid
Target positive class: two_year_recid == 0 / no recidivism
Protected column: race
Pr